## PingOne Overview

PingOne is a cloud-based identity and access management service that provides secure identity solutions for enterprises, enabling seamless authentication and authorization across applications and services.

Key Features:
* **Single Sign-On (SSO)** - Users authenticate once to access multiple applications
* **Multi-Factor Authentication (MFA)** - Enhanced security through additional verification methods
* **Adaptive Authentication** - Risk-based authentication policies based on user behavior and context
* **Universal Directory** - Centralized user management and profile synchronization
* **API Access Management** - OAuth 2.0 and OpenID Connect support for API security

## Learning Objective

Outbound authentication enables agents to access external APIs on behalf of authenticated users. In this notebook, an agent will call the PingOne Management API to retrieve the user's profile information, demonstrating the delegated authorization pattern.

## Tutorial Architecture

<figure style="width: 70%;">
    <img src="/images/outbound_auth_diagram.png" style="width: 70%; height: auto;" alt="Outbound Auth Architecture">
</figure>

## Prerequisites

- PingOne environment
- PingOne admin rights to create new applications
- Python 3.10+
- AWS credentials
- AWS IAM rights to create AgentCore resources
- Amazon Bedrock AgentCore SDK
- Strands Agents
- FastAPI and Uvicorn (for local OAuth2 callback server)
- AWS region that supports Bedrock AgentCore. Refer https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html

## Step 1: Setup PingOne for Outbound Auth

### 1.1: Create a New Application

**This application will be used by the agent to access PingOne APIs on behalf of users via the authorization code flow.**

1. Login to your PingOne environment
2. Select **Applications**, then **Applications**, and add an application
3. Select **OIDC Web App** as the **Application Type**
4. Configure the application:
    - For **Application Name**, enter **AgentCore Outbound Auth**
    - For **Token Auth Method**, select **Client Secret Basic**
    - For **Grant Type**, select **Authorization Code**
    - For **Redirect URIs**, add the local callback URL for testing:
      `http://localhost:9090/oauth2/callback`
    - For **Scopes**, enable **openid**, **profile**, and **email**

<figure>
    <img src="/images/outbound_application_config.png">
</figure>

### 1.2: Note Your Application Credentials

After creating the application, note the following values for use in this notebook:
- **Client ID**
- **Client Secret**
- **Environment ID** (found in your PingOne environment settings)
- **Authorization Endpoint**: `https://auth.pingone.com/{env_id}/as/authorize`
- **Token Endpoint**: `https://auth.pingone.com/{env_id}/as/token`

## Step 2: Notebook Setup and AWS Configuration

### 2.1: Install Dependencies for Notebook

In [ ]:
# Install required packages from requirements.txt.
%pip install --force-reinstall -U -r ../requirements.txt --quiet

### 2.2: Load Environment Variables for Notebook

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

PINGONE_CLIENT_ID = os.environ['OUTBOUND_PINGONE_CLIENT_ID']
PINGONE_CLIENT_SECRET = os.environ['OUTBOUND_PINGONE_CLIENT_SECRET']
PINGONE_ENV_ID = os.environ['OUTBOUND_PINGONE_ENV_ID']
PINGONE_AUTHORIZE_URL = f'https://auth.pingone.com/{PINGONE_ENV_ID}/as/authorize'
PINGONE_TOKEN_URL = f'https://auth.pingone.com/{PINGONE_ENV_ID}/as/token'
PINGONE_API_BASE = f'https://api.pingone.com/v1/environments/{PINGONE_ENV_ID}'

### 2.3: Discover AWS Environment Identity

In [ ]:
import boto3
from boto3.session import Session

# Create an AWS session from environment variables or ~/.aws/credentials.
boto_session = Session()
region = boto_session.region_name

# Retrieve account ID for verification.
sts = boto3.client('sts')
account_id = sts.get_caller_identity().get('Account')

print(f'📍 Deploying to AWS Account ID {account_id} in {region}.')

### 2.4: Configure Resource Names

Define names for the AWS resources that will be created. Customize these as needed.

In [ ]:
from oauth2_callback_server import get_oauth2_callback_url

# OAuth2 Credential Provider: Registers PingOne as an outbound auth provider with AgentCore.
CREDENTIAL_PROVIDER_NAME = 'pingone-outbound-provider'

# Workload Identity: Manages the OAuth2 callback URLs for the credential provider.
WORKLOAD_IDENTITY_NAME = 'pingone-outbound-workload'

# Callback URL: Local server for testing, receives the authorization code from PingOne.
OAUTH_CALLBACK_URL = get_oauth2_callback_url()
print(f'📍 Using callback URL: {OAUTH_CALLBACK_URL}')

## Step 3: Register PingOne as an OAuth2 Credential Provider

### 3.1: Create the Identity Client

Initialize the AgentCore Identity client to manage credential providers.

In [ ]:
from bedrock_agentcore.services.identity import IdentityClient

identity_client = IdentityClient(region=region)

### 3.2: Register the OAuth2 Credential Provider

Tell AgentCore Identity how to authenticate with PingOne on behalf of users.

In [ ]:
# Register PingOne as a custom OAuth2 credential provider.
try:
    identity_client.create_oauth2_credential_provider(req={
        'name': CREDENTIAL_PROVIDER_NAME,
        'credentialProviderVendor': 'CustomOauth2',
        'oauth2ProviderConfigInput': {
            'customOauth2ProviderConfig': {
                'clientId': PINGONE_CLIENT_ID,
                'clientSecret': PINGONE_CLIENT_SECRET,
                'authorizeEndpoint': PINGONE_AUTHORIZE_URL,
                'tokenEndpoint': PINGONE_TOKEN_URL
            }
        }
    })
    print(f'✅ Created credential provider: {CREDENTIAL_PROVIDER_NAME}')
except Exception as e:
    if 'already exists' in str(e).lower():
        print(f'✅ Credential provider already exists: {CREDENTIAL_PROVIDER_NAME}')
    else:
        raise e

### 3.3: Configure Workload Identity

Set up the workload identity with allowed callback URLs.

In [ ]:
# Create or update the workload identity with allowed callback URLs.
try:
    identity_client.create_workload_identity(name=WORKLOAD_IDENTITY_NAME)
    print(f'✅ Created workload identity: {WORKLOAD_IDENTITY_NAME}')
except Exception as e:
    if 'already exists' in str(e).lower():
        print(f'✅ Workload identity already exists: {WORKLOAD_IDENTITY_NAME}')
    else:
        raise e

# Whitelist the callback URL.
identity_client.update_workload_identity(
    name=WORKLOAD_IDENTITY_NAME,
    allowed_resource_oauth_2_return_urls=[OAUTH_CALLBACK_URL]
)
print(f'✅ Whitelisted callback URL: {OAUTH_CALLBACK_URL}')

## Step 4: Define Agent with Outbound Auth Tool

### 4.1: Create the PingOne API Tool

Define a tool that calls the PingOne Management API to retrieve user information. The `@requires_access_token` decorator handles the OAuth flow automatically.

In [ ]:
import requests
import json
from strands import Agent, tool
from bedrock_agentcore.identity.auth import requires_access_token

@tool
@requires_access_token(
    provider_name=CREDENTIAL_PROVIDER_NAME,
    scopes=['openid', 'profile', 'email'],
    callback_url=OAUTH_CALLBACK_URL,
    on_auth_url=lambda url: print(f'\n🔐 Please open this URL in your browser to authorize:\n{url}\n'),
)
def get_my_pingone_profile(access_token: str) -> str:
    """
    Retrieves the current user's profile information from PingOne.
    Call this when the user asks about their identity or profile.
    """
    # Use the userinfo endpoint to get the authenticated user's profile.
    userinfo_url = f'https://auth.pingone.com/{PINGONE_ENV_ID}/as/userinfo'
    
    response = requests.get(
        userinfo_url,
        headers={'Authorization': f'Bearer {access_token}'}
    )
    
    if response.status_code == 200:
        return json.dumps(response.json(), indent=2)
    else:
        return f'Error retrieving profile: {response.status_code} - {response.text}'

### 4.2: Create the Agent

Create a Strands agent with the PingOne tool.

In [ ]:
agent = Agent(
    tools=[get_my_pingone_profile],
    system_prompt='You help users retrieve information about their identity from PingOne.'
)

## Step 5: Test the Agent

### 5.1: Invoke the Agent

Ask the agent to retrieve your PingOne profile. This will:
1. Start a local OAuth2 callback server
2. Trigger the authorization flow when the agent calls the tool
3. Display a URL for you to authenticate in your browser
4. Capture the authorization code and exchange it for tokens

In [ ]:
import sys
import subprocess
from oauth2_callback_server import (
    wait_for_oauth2_server_to_be_ready,
    store_user_id_in_oauth2_callback_server,
)

# Start the local OAuth2 callback server in a subprocess.
oauth2_server_cmd = [sys.executable, 'oauth2_callback_server.py', '--region', region]
oauth2_server_process = subprocess.Popen(oauth2_server_cmd)

try:
    # Wait for the server to be ready.
    if not wait_for_oauth2_server_to_be_ready():
        raise RuntimeError('Failed to start OAuth2 callback server')
    print('✅ OAuth2 callback server is ready')
    
    # Register a user ID for session binding.
    user_id = 'notebook-user'
    store_user_id_in_oauth2_callback_server(user_id)
    print(f'✅ Registered user ID: {user_id}')
    
    # Invoke the agent - this will trigger the OAuth flow.
    print('=' * 60)
    print('TEST: Retrieve PingOne Profile')
    print('=' * 60)
    
    result = agent('What is my PingOne profile information?')
    
    print('\n📋 Agent Response:')
    print(result)
    
finally:
    # Clean up the callback server.
    oauth2_server_process.terminate()
    print('\n✅ OAuth2 callback server stopped')

## Conclusion and Cleanup

### Resource Cleanup

Clean up the resources created in this tutorial.

In [ ]:
# Delete the credential provider and workload identity.
print('🗑️ Cleaning up resources...')

try:
    identity_client.delete_oauth2_credential_provider(name=CREDENTIAL_PROVIDER_NAME)
    print(f'✅ Deleted credential provider: {CREDENTIAL_PROVIDER_NAME}')
except Exception as e:
    print(f'⚠️ Could not delete credential provider: {e}')

try:
    identity_client.delete_workload_identity(name=WORKLOAD_IDENTITY_NAME)
    print(f'✅ Deleted workload identity: {WORKLOAD_IDENTITY_NAME}')
except Exception as e:
    print(f'⚠️ Could not delete workload identity: {e}')

print('✨ Cleanup finished.')

### Conclusion

This notebook demonstrated how to:
1. **Setup PingOne** - Configure an application for authorization code flow with a local callback URL
2. **Register Credential Provider** - Tell AgentCore how to authenticate with PingOne
3. **Create Outbound Auth Tool** - Use `@requires_access_token` with a local callback server
4. **Test the Flow** - Start the callback server, trigger OAuth, and retrieve user profile data

### Key Learnings

- **Outbound Authentication:** Agents can access external APIs on behalf of users with delegated authorization
- **OAuth2 Credential Providers:** AgentCore Identity manages the OAuth flow and token storage
- **Local Callback Server:** Required for local testing to capture the authorization code from the identity provider
- **Session Binding:** User identity must be registered with the callback server before starting the OAuth flow
- **Just-in-Time Authorization:** Users only authorize when the agent needs to access their data

### Next Steps

- Deploy to AgentCore Runtime (uses managed callback URL instead of local server)
- Add more PingOne API tools (list applications, manage groups)
- Combine with inbound auth for end-to-end security
- Explore other OAuth2 providers (Google, GitHub, Salesforce)